# Eksperimen Random Forest: label per kelurahan dari titik banjir (Solusi B)

Notebook ini adalah eksperimen baru. Notebook baseline `ml/flood_risk_training.ipynb` tidak diubah.

**Keputusan metodologis yang diterapkan:**
- Unit analisis: kelurahan Kota Bekasi (48 poligon dari `ADMINISTRASIDESA_AR_25K`).
- Spatial join memakai geometri titik sebagai acuan. Atribut `kecamatan` sumber hanya dicatat untuk validasi.
- Label per kelurahan = severity tertinggi dari semua titik di dalam poligon (semua tahun 2020 sampai 2026).
- Kelurahan tanpa titik banjir **tidak diberi label**. Tidak ada label default `sedang`.
- Varian A: titik `low` dikeluarkan dari kelurahan berlabel `low` tertinggi. Varian B: `low` digabung ke `sedang`.
- Evaluasi utama: Leave-One-Kecamatan-Out (LOGO) dengan 12 kecamatan Kota Bekasi sebagai grup.
- Scaler dan Random Forest berada dalam satu pipeline, sehingga normalisasi di-fit hanya pada data training setiap fold.
- Hyperparameter tetap seperti baseline (200 pohon, `max_depth=8`, `class_weight='balanced'`). Tidak ada tuning.

**Keterbatasan yang harus dibaca bersama hasil:** asal atribut `severity` belum terverifikasi dari metadata. Jumlah sampel kecil (27 dan 33 kelurahan). Titik banjir pada sumber bisa berulang di lokasi yang sama pada tahun berbeda.

In [ ]:
import json, sys, warnings
import numpy as np, pandas as pd, geopandas as gpd
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneGroupOut, StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix
warnings.filterwarnings('ignore')
print('python', sys.version.split()[0], '| sklearn', sklearn.__version__, '| numpy', np.__version__, '| geopandas', gpd.__version__)
FEATURE_COLS = ['shape_area','shape_leng','compactness','elev_mean','elev_min','sungai_density','dist_to_river','pct_pemukiman','pct_sawah']
SEED = 42

In [ ]:
points = gpd.read_file('../data_banjir.geojson').to_crs('EPSG:4326')
polys = gpd.read_file('../topo/geojson_kota_bekasi/ADMINISTRASIDESA_AR_25K.geojson').to_crs('EPSG:4326')
feat = pd.read_csv('features_kelurahan_82.csv')
kota = polys[polys['WADMKK'] == 'Kota Bekasi'][['NAMOBJ', 'WADMKC', 'geometry']].copy()
print('titik banjir:', len(points), '| poligon total:', len(polys), '| poligon Kota Bekasi:', len(kota))
print('baris fitur:', len(feat), '| kolom fitur hilang:', [c for c in FEATURE_COLS if c not in feat.columns])

In [ ]:
points['row_idx'] = range(len(points))
pj = gpd.sjoin(points, kota, how='inner', predicate='within')
assert pj['row_idx'].nunique() == len(points), 'ada titik di luar poligon Kota Bekasi'
print('titik yang jatuh ke poligon kelurahan Kota Bekasi:', pj['row_idx'].nunique(), 'dari', len(points))
print('kelurahan dengan minimal satu titik:', pj['NAMOBJ'].nunique(), 'dari', len(kota))

In [ ]:
RANK = {'low': 1, 'medium': 2, 'high': 3}
severity_list = pj.groupby('NAMOBJ')['severity'].apply(list).rename('severity_list')
severity_max_raw = pj.groupby('NAMOBJ')['severity'].apply(lambda s: max(s, key=RANK.get)).rename('severity_max_raw')
lab = kota[['NAMOBJ', 'WADMKC']].drop_duplicates('NAMOBJ').set_index('NAMOBJ').join([severity_list, severity_max_raw]).reset_index()
lab['n_points'] = lab['severity_list'].apply(lambda x: len(x) if isinstance(x, list) else 0)
# Varian A: low dikeluarkan. Varian B: low digabung ke sedang. Kelurahan tanpa titik tetap NaN (tidak berlabel).
lab['label_A'] = lab['severity_max_raw'].map({'high': 'tinggi', 'medium': 'sedang'})
lab['label_B'] = lab['severity_max_raw'].map({'high': 'tinggi', 'medium': 'sedang', 'low': 'sedang'})
print('distribusi severity tertinggi per kelurahan:')
print(lab['severity_max_raw'].value_counts(dropna=False).to_string())

In [ ]:
key = {str(k).upper(): k for k in feat['kelurahan']}
lab['kel_key'] = lab['NAMOBJ'].str.upper().map(key)
assert lab['kel_key'].notna().all(), 'ada kelurahan tanpa baris fitur'
df = lab.merge(feat.rename(columns={'kelurahan': 'kel_key'}), on='kel_key', how='left', validate='one_to_one')
assert df[FEATURE_COLS].notna().all().all(), 'ada fitur hilang'
assert df.loc[df['n_points'] == 0, ['label_A', 'label_B']].isna().all().all(), 'kelurahan tanpa titik mendapat label'
print('kelurahan dengan fitur lengkap:', len(df))
print('kelurahan tanpa titik (tidak berlabel, tidak dianggap bebas banjir):', int((df['n_points'] == 0).sum()))
print('label_A:', df['label_A'].value_counts().to_dict())
print('label_B:', df['label_B'].value_counts().to_dict())
df.to_csv('dataset_eksperimen_per_kelurahan.csv', index=False, encoding='utf-8')

In [ ]:
def evaluate_logo(data, label_col, seed=SEED):
    d = data[data[label_col].notna()].reset_index(drop=True)
    X = d[FEATURE_COLS].to_numpy(float)
    y = d[label_col].to_numpy()
    groups = d['WADMKC'].to_numpy()
    classes = sorted(set(y))
    logo = LeaveOneGroupOut()
    base_pred = np.empty(len(y), dtype=object)
    for tr, te in logo.split(X, y, groups):
        base_pred[te] = pd.Series(y[tr]).value_counts().idxmax()
    pipe = make_pipeline(StandardScaler(), RandomForestClassifier(n_estimators=200, max_depth=8, class_weight='balanced', random_state=seed))
    rf_pred = cross_val_predict(pipe, X, y, cv=logo, groups=groups)
    def metrics(p):
        return dict(accuracy=accuracy_score(y, p), balanced_accuracy=balanced_accuracy_score(y, p), macro_f1=f1_score(y, p, average='macro'))
    per_group = pd.DataFrame({'kecamatan': groups, 'aktual': y, 'prediksi': rf_pred}).groupby('kecamatan').apply(
        lambda g: pd.Series({'n': len(g), 'akurasi': (g['aktual'] == g['prediksi']).mean()}))
    return dict(n=len(d), n_groups=len(set(groups)), classes=classes, class_counts={k: int(v) for k, v in pd.Series(y).value_counts().items()},
                baseline=metrics(base_pred), rf_logo=metrics(rf_pred),
                confusion=confusion_matrix(y, rf_pred, labels=classes).tolist(), per_group=per_group)

res_A = evaluate_logo(df, 'label_A')
res_B = evaluate_logo(df, 'label_B')
for name, r in [('A (low dikeluarkan)', res_A), ('B (low -> sedang)', res_B)]:
    print(f"\n== Varian {name} | n={r['n']} | kelas={r['class_counts']} | kelompok kecamatan={r['n_groups']}")
    print('baseline mayoritas :', {k: round(v, 3) for k, v in r['baseline'].items()})
    print('RF LOGO            :', {k: round(v, 3) for k, v in r['rf_logo'].items()})
    print('confusion [baris=aktual, kolom=prediksi] kelas', r['classes'], '->', r['confusion'])
    print(r['per_group'].round(3).to_string())

In [ ]:
seed_rows = []
for label_col in ['label_A', 'label_B']:
    for s in range(5):
        r = evaluate_logo(df, label_col, seed=s)
        seed_rows.append({'varian': label_col, 'seed': s, **{f'rf_{k}': v for k, v in r['rf_logo'].items()}, **{f'base_{k}': v for k, v in r['baseline'].items()}})
seed_df = pd.DataFrame(seed_rows)
summary = seed_df.groupby('varian')[['rf_balanced_accuracy', 'rf_macro_f1', 'base_balanced_accuracy']].agg(['mean', 'std']).round(3)
print('Stabilitas LOGO terhadap seed RF (5 seed):')
print(summary.to_string())

In [ ]:
def evaluate_random_reference(data, label_col, seed=SEED):
    d = data[data[label_col].notna()].reset_index(drop=True)
    X = d[FEATURE_COLS].to_numpy(float); y = d[label_col].to_numpy()
    pipe = make_pipeline(StandardScaler(), RandomForestClassifier(n_estimators=200, max_depth=8, class_weight='balanced', random_state=seed))
    p = cross_val_predict(pipe, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=seed))
    return dict(accuracy=accuracy_score(y, p), balanced_accuracy=balanced_accuracy_score(y, p), macro_f1=f1_score(y, p, average='macro'))
print('REFERENSI SAJA, BUKAN KLAIM VALIDASI (kelurahan se-kecamatan dapat berbagi label):')
for col in ['label_A', 'label_B']:
    print(col, {k: round(v, 3) for k, v in evaluate_random_reference(df, col).items()})

In [ ]:
def jsonable(r):
    out = {k: v for k, v in r.items() if k != 'per_group'}
    out['per_group'] = {k: {kk: float(vv) for kk, vv in v.items()} for k, v in r['per_group'].to_dict('index').items()}
    return out
with open('results_logo.json', 'w', encoding='utf-8') as f:
    json.dump({'varian_A': jsonable(res_A), 'varian_B': jsonable(res_B), 'seed_stability': seed_rows}, f, ensure_ascii=False, indent=2, default=float)
print('hasil disimpan ke results_logo.json')